# BGE-M3 cho pilot Toán tiếng Việt — Kaggle / Colab GPU

Notebook **suy luận embedding, không fine-tune/train BGE-M3**. Tạo vector dense 1024 chiều cho 40 đoạn giải thích/gợi ý, đánh giá 90 query nháp với metadata/BM25, rồi xuất JSON để nhập về dự án. Giữ tên file `bge_m3_colab.ipynb` để dùng chung trên hai nền tảng.

**Chuẩn bị tại máy:** chạy `python scripts/exports/prepare_retrieval_colab.py` trong dự án để tạo `data/exports/pilot-retrieval-input.zip`. Không tải toàn bộ repo, `.env`, API key, dataset ASSISTments hay lời giải chuẩn.

**Kaggle:** import notebook này; upload ZIP thành Dataset **Private**, rồi **Add Input** vào notebook. Kaggle có thể giải nén ZIP; notebook nhận cả ZIP nguyên và thư mục đã giải nén. Trong Settings, bật **Internet**, chọn GPU (T4 nếu có). Chọn **Save Version → Save & Run All** để chạy nền từ đầu đến cuối; chờ xác nhận phiên bắt đầu trước khi tắt máy. Kết quả là `/kaggle/working/pilot-retrieval-bge-m3.zip`, tải trong **Output** của phiên đã hoàn thành. Chạy nền vẫn chịu giới hạn tài nguyên của Kaggle.

**Colab:** chọn **Runtime → Change runtime type → T4 GPU**, chạy các ô lần lượt và upload ZIP khi được yêu cầu. Ô cuối tải kết quả về máy.

Model công khai được tải từ Hugging Face, cần Internet và dung lượng vài GB. Nội dung/nhãn đang là nháp nội bộ; kết quả chưa chứng minh hiệu quả với học sinh.


In [1]:
from pathlib import Path
from zipfile import ZipFile
import hashlib, io, json, sys, uuid

# Kaggle: thường để None cho tự tìm. Nếu gắn nhiều gói, điền đường dẫn
# ZIP hoặc thư mục chứa manifest.json trong /kaggle/input/ rồi Save & Run All.
INPUT_PATH = None
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
RUNTIME_PLATFORM = 'kaggle' if KAGGLE_INPUT_ROOT.is_dir() else 'colab'
WORK_ROOT = Path('/kaggle/working') if RUNTIME_PLATFORM == 'kaggle' else Path('/content')
CACHE_ROOT = Path('/kaggle/temp') if RUNTIME_PLATFORM == 'kaggle' else Path('/content')

CODE_FILES = ['backend/app/__init__.py','backend/app/contracts.py','backend/app/curriculum.py',
              'backend/app/pilot_contracts.py','backend/app/retrieval_contracts.py','backend/app/retrieval.py',
              'backend/app/local_embedding.py','research/retrieval/evaluate.py','research/retrieval/requirements.txt']
INPUT_FILES = CODE_FILES + ['corpus.json','queries.json','manifest.json']
MAX_INPUT_BYTES = 5*1024*1024

def read_archive(archive_bytes):
    if len(archive_bytes) > MAX_INPUT_BYTES:
        raise ValueError('Input ZIP quá lớn.')
    expected = {'pilot-retrieval/'+name for name in INPUT_FILES}
    with ZipFile(io.BytesIO(archive_bytes)) as bundle:
        if set(bundle.namelist()) != expected or len(bundle.namelist()) != len(expected):
            raise ValueError('ZIP thiếu file hoặc chứa file ngoài danh sách được phép.')
        if sum(i.file_size for i in bundle.infolist()) > MAX_INPUT_BYTES:
            raise ValueError('Nội dung ZIP quá lớn.')
        return {name:bundle.read('pilot-retrieval/'+name) for name in INPUT_FILES}

def read_kaggle_input(source):
    source = Path(source)
    if source.is_file():
        if source.stat().st_size > MAX_INPUT_BYTES:
            raise ValueError('Input ZIP quá lớn.')
        return read_archive(source.read_bytes())
    if not source.is_dir():
        raise ValueError('INPUT_PATH không tồn tại: '+str(source))
    if not (source/'manifest.json').is_file() and (source/'pilot-retrieval/manifest.json').is_file():
        source = source/'pilot-retrieval'
    if {p.relative_to(source).as_posix() for p in source.rglob('*') if p.is_file()} != set(INPUT_FILES):
        raise ValueError('Thư mục input thiếu file hoặc chứa file ngoài danh sách được phép.')
    if sum((source/name).stat().st_size for name in INPUT_FILES) > MAX_INPUT_BYTES:
        raise ValueError('Nội dung input quá lớn.')
    return {name:(source/name).read_bytes() for name in INPUT_FILES}

def find_kaggle_input(root):
    archives = sorted(root.rglob('pilot-retrieval-input.zip'))
    candidates = archives or sorted(p.parent for p in root.rglob('manifest.json')
                                   if (p.parent/'corpus.json').is_file() and (p.parent/'queries.json').is_file())
    if not candidates:
        raise ValueError('Chưa có gói input. Upload pilot-retrieval-input.zip thành Dataset Private và Add Input vào notebook.')
    if len(candidates) != 1:
        raise ValueError('Có nhiều gói input; điền INPUT_PATH để chọn đúng gói: '+str(candidates))
    return candidates[0]

if RUNTIME_PLATFORM == 'kaggle':
    input_source = Path(INPUT_PATH) if INPUT_PATH else find_kaggle_input(KAGGLE_INPUT_ROOT)
    print('Kaggle input:', input_source)
    raw = read_kaggle_input(input_source)
else:
    from google.colab import files
    uploaded = files.upload()
    if set(uploaded) != {'pilot-retrieval-input.zip'}:
        raise ValueError('Chỉ chọn một file pilot-retrieval-input.zip đã chuẩn bị từ dự án.')
    raw = read_archive(uploaded['pilot-retrieval-input.zip'])

manifest = json.loads(raw['manifest.json'])
if manifest['schema_version'] != '1.0.0' or set(manifest['sha256']) != set(CODE_FILES+['corpus.json','queries.json']):
    raise ValueError('Manifest không hợp lệ.')
for name, expected_sha in manifest['sha256'].items():
    if hashlib.sha256(raw[name]).hexdigest() != expected_sha:
        raise ValueError('SHA256 sai: '+name)
WORK_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR = WORK_ROOT/('pilot-retrieval-'+uuid.uuid4().hex)
WORK_DIR.mkdir()
for name, value in raw.items():
    target = WORK_DIR/name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
print('Nền tảng:', RUNTIME_PLATFORM, '| Đã kiểm tra gói dữ liệu:', WORK_DIR)


Kaggle input: /kaggle/input/datasets/tintthn/pilot-retrieval-input/pilot-retrieval
Nền tảng: kaggle | Đã kiểm tra gói dữ liệu: /kaggle/working/pilot-retrieval-354de82920bc4706b2ea403660ec0e03


## Cài thư viện và kiểm tra GPU
Giữ PyTorch có sẵn của Kaggle/Colab. Model, revision, pooling và độ dài phải giống backend. Ô cài đặt có thể mất vài phút. Trên Kaggle cần bật Internet trước khi Save & Run All. Nếu môi trường yêu cầu restart, restart rồi chạy lại từ đầu; với Kaggle tạo phiên Save & Run All mới sau khi sửa cấu hình.


In [2]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK_DIR/'research/retrieval/requirements.txt')])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.0/488.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


0

In [3]:
sys.path.insert(0, str(WORK_DIR/'backend'))
sys.path.insert(0, str(WORK_DIR))
import torch, time, importlib.metadata, platform
from app.local_embedding import ENCODER, load_model, encode_texts
from app.retrieval import embedding_text, build_index, corpus_hash, retrieve
from app.retrieval_contracts import RetrievalRequest
from research.retrieval.evaluate import load_inputs, save_query_vectors, evaluate

if not torch.cuda.is_available():
    raise RuntimeError('Chưa có GPU. Kaggle: Settings → Accelerator → GPU; Colab: Runtime → Change runtime type → T4 GPU.')
chunks, dataset = load_inputs(WORK_DIR)
if manifest['encoder'] != ENCODER or manifest['corpus_sha256'] != corpus_hash(chunks):
    raise ValueError('Encoder/corpus không khớp manifest.')
environment = {'runtime_platform':RUNTIME_PLATFORM, 'python':platform.python_version(), 'torch':torch.__version__, 'cuda':torch.version.cuda,
               'gpu':torch.cuda.get_device_name(0), 'precision':'float16', 'batch_size':8,
               'packages':{name:importlib.metadata.version(name) for name in ['sentence-transformers','transformers','pydantic']}}
print(json.dumps({'chunks':len(chunks),'queries':len(dataset['queries']), 'encoder':ENCODER, 'environment':environment}, ensure_ascii=False, indent=2))

{
  "chunks": 40,
  "queries": 90,
  "encoder": {
    "model_id": "BAAI/bge-m3",
    "revision": "5617a9f61b028005a4858fdac845db406aefb181",
    "dimensions": 1024,
    "max_length": 512,
    "pooling": "model_sentence_transformers",
    "normalize": true,
    "query_prefix": "",
    "passage_prefix": ""
  },
  "environment": {
    "runtime_platform": "kaggle",
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "cuda": "12.8",
    "gpu": "Tesla T4",
    "precision": "float16",
    "batch_size": 8,
    "packages": {
      "sentence-transformers": "5.1.2",
      "transformers": "4.57.6",
      "pydantic": "2.13.5"
    }
  }
}


## Tải BGE-M3 và tạo vector
Cache model nằm trong `/kaggle/temp/bge-m3-model-cache` trên Kaggle hoặc `/content/bge-m3-model-cache` trên Colab. Cache tạm có thể mất khi phiên kết thúc; không đóng gói model weights vào Output. Dùng batch 8, max_length 512 và FP16. Kiểm tra tokenizer sẽ báo lỗi khi văn bản quá dài, không âm thầm cắt nội dung. Nếu hết VRAM, giảm `BATCH_SIZE` xuống 4 hoặc 2 và chạy lại (Kaggle: Save & Run All phiên mới).


In [4]:
started = time.perf_counter()
model = load_model(download=True, device='cuda', cache_folder=CACHE_ROOT/'bge-m3-model-cache')
torch.cuda.synchronize()
timings = {'model_load_seconds':round(time.perf_counter()-started, 3)}
print('Model đã tải:', ENCODER['model_id'], ENCODER['revision'])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model đã tải: BAAI/bge-m3 5617a9f61b028005a4858fdac845db406aefb181


In [5]:
BATCH_SIZE = 8
if not 1 <= BATCH_SIZE <= 8:
    raise ValueError('BATCH_SIZE phải trong khoảng 1–8.')
environment['batch_size'] = BATCH_SIZE
torch.cuda.reset_peak_memory_stats()
def timed_encode(texts, name):
    torch.cuda.synchronize()
    started = time.perf_counter()
    result = encode_texts(model, texts, batch_size=BATCH_SIZE)
    torch.cuda.synchronize()
    timings[name] = round(time.perf_counter()-started, 3)
    return result

corpus_vectors = timed_encode([embedding_text(c) for c in chunks], 'corpus_encode_seconds')
query_vectors = timed_encode([q['request']['query'] for q in dataset['queries']], 'query_encode_seconds')
timings['peak_gpu_memory_mib'] = round(torch.cuda.max_memory_allocated()/1024**2, 1)
def write_json(name, value):
    (WORK_DIR/name).write_text(json.dumps(value, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
write_json('index.json', build_index(chunks, corpus_vectors, environment=environment))
write_json('query-vectors.json', save_query_vectors(dataset['queries'], query_vectors))
print(json.dumps(timings, indent=2))

{
  "model_load_seconds": 76.867,
  "corpus_encode_seconds": 1.537,
  "query_encode_seconds": 0.272,
  "peak_gpu_memory_mib": 1105.1
}


## Đánh giá cùng bộ lọc nội dung
Tính Recall@3, Hit@3, nDCG@3 cho metadata, lexical BM25 và semantic. Semantic không được fallback trong đánh giá; thiếu/sai vector sẽ báo lỗi. 80 query có relevance nháp, 10 query kiểm tra scope không có nội dung; chưa có kiểm thử tổng quát câu hỏi ngoài miền. Các paraphrase có liên quan nhau, không phải 90 mẫu độc lập.


In [6]:
report = evaluate(WORK_DIR, semantic=True)
write_json('evaluation.json', report)
for method, scores in report['methods'].items():
    print(f"{method:10} Recall@3={scores['recall_at_3']:.3f}  Hit@3={scores['hit_at_3']:.3f}  nDCG@3={scores['ndcg_at_3']:.3f}")
print('Chi tiết mỗi query và mỗi concept trong evaluation.json; không mặc định semantic tốt hơn baseline.')

metadata   Recall@3=0.606  Hit@3=0.613  nDCG@3=0.488
lexical    Recall@3=0.840  Hit@3=0.963  nDCG@3=0.852
semantic   Recall@3=0.875  Hit@3=0.938  nDCG@3=0.854
Chi tiết mỗi query và mỗi concept trong evaluation.json; không mặc định semantic tốt hơn baseline.


## Thử một câu hỏi
Chỉ nhập câu hỏi học thuật, không nhập thông tin cá nhân. Đây là xem các gợi ý ứng viên được phép theo mức hỗ trợ, chưa phải Agent quyết định can thiệp.


In [7]:
query = 'Em thường quên nhân số đứng ngoài ngoặc với hạng tử thứ hai, nên kiểm tra thế nào?'
request = RetrievalRequest(query=query, concept_ids=sorted({c.concept_id for c in chunks}),
                           purpose='hint', hint_level=1, top_k=3, method='semantic')
query_vector = encode_texts(model, [query], batch_size=1)[0]
result = retrieve(chunks, request, authoring=True, index_path=WORK_DIR/'index.json',
                  query_vector=query_vector, allow_fallback=False)
for hit in result.hits:
    print(hit.chunk.content_id, round(hit.score, 4), hit.chunk.text, sep=' | ')

A02-hint-1 | 0.6505 | Có bao nhiêu hạng tử trong ngoặc cần được nhân?
N02-hint-1 | 0.6271 | Em có thể xác định dấu trước khi tính độ lớn không?
E02-hint-1 | 0.5246 | Em muốn tách phần chứa ẩn hay hằng số trước?


## Xuất kết quả về máy
ZIP chỉ chứa JSON, không chứa model weights hay code chạy. Trên Kaggle, file nằm trực tiếp ở `/kaggle/working/pilot-retrieval-bge-m3.zip`; sau Save & Run All thành công, tải trong **Output** của phiên đã hoàn thành. Không gọi hàm download tương tác khi chạy nền. Trên Colab, ô này tải ZIP về trình duyệt.

Đưa ZIP vào `data/exports/` rồi chạy lệnh import ở hướng dẫn dự án. Giữ nguyên tên các file trong ZIP.


In [8]:
from zipfile import ZIP_DEFLATED
ARTIFACT_FILES = ['index.json','query-vectors.json','evaluation.json','corpus.json','queries.json']
run_manifest = {'schema_version':'1.0.0', 'encoder':ENCODER, 'corpus_sha256':corpus_hash(chunks),
                'environment':environment, 'timings':timings,
                'input_manifest_sha256':hashlib.sha256(raw['manifest.json']).hexdigest(),
                'sha256':{name:hashlib.sha256((WORK_DIR/name).read_bytes()).hexdigest() for name in ARTIFACT_FILES}}
write_json('run-manifest.json', run_manifest)
output_zip = WORK_ROOT/'pilot-retrieval-bge-m3.zip'
with ZipFile(output_zip, 'w', ZIP_DEFLATED) as bundle:
    for name in ARTIFACT_FILES+['run-manifest.json']:
        bundle.write(WORK_DIR/name, name)
if RUNTIME_PLATFORM == 'colab':
    files.download(str(output_zip))
else:
    print('Kaggle: sau khi phiên hoàn thành, tải ZIP này trong Output:', output_zip)
print('Đã xuất ZIP:', output_zip)
print('Xem docs/implementation/curriculum/grounding.v1.md để nhập vào dự án.')

Kaggle: sau khi phiên hoàn thành, tải ZIP này trong Output: /kaggle/working/pilot-retrieval-bge-m3.zip
Đã xuất ZIP: /kaggle/working/pilot-retrieval-bge-m3.zip
Xem docs/implementation/curriculum/grounding.v1.md để nhập vào dự án.


## Sau Kaggle / Colab
Trong thư mục dự án:
```powershell
.\.venv\Scripts\python.exe scripts/exports/import_retrieval_artifact.py data/exports/pilot-retrieval-bge-m3.zip
```
Index đã có vector **nội dung**; backend vẫn cần BGE-M3 cùng revision để biến **câu hỏi mới** thành vector. Cài dependencies và tải model một lần ở máy chạy backend theo hướng dẫn. API không tự tải model trong request. Kaggle/Colab là môi trường tạo index và đánh giá, không dùng làm server sản phẩm lâu dài.

Nguồn: [BGE-M3](https://huggingface.co/BAAI/bge-m3), [Sentence Transformers](https://sbert.net/docs/package_reference/sentence_transformer/model.html), [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks).
